In [1]:
# Đọc dữ liệu từ Silver
df_sales = spark.table("silver_sales")
df_exchange_rate = spark.table("silver_exchange_rate")

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F

# Tạo DataFrame dim_date
df_dim_date = (
    df_sales
    .select(F.to_date(F.col("order_date")).alias("full_date"))
    .dropDuplicates()
    .filter(F.col("full_date").isNotNull())
    .withColumn("date_key", F.date_format(F.col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", F.year(F.col("full_date")))
    .withColumn("month", F.month(F.col("full_date")))
    .withColumn("day", F.dayofmonth(F.col("full_date")))
    .withColumn("quarter", F.quarter(F.col("full_date")))
    .select(
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "day"
    )
)

display(df_dim_date.limit(10))

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a82ea6e3-d7fb-4a8f-bbda-fe100a1c61c7)

In [3]:
# Tạo DataFrame dim_location
df_dim_location = (
    df_sales
    .select(F.trim(F.col("location")).alias("city"))
    .filter(F.col("city").isNotNull())
    .dropDuplicates()
    .withColumn(
        "location_id",
        F.md5(F.lower(F.col("city")))
    )
    .select(
        "location_id",
        "city"
    )
)

display(df_dim_location.limit(10))

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9261d409-143c-481f-9156-2635d485f0f1)

In [4]:
# Tạo DataFrame dim_customer
df_dim_customer = (
    df_sales
    .select(
        F.trim(F.col("customer_name")).alias("customer_name"),
        F.lower(F.trim(F.col("customer_email"))).alias("customer_email"),
        F.trim(F.col("customer_phone")).alias("customer_phone"),
        F.col("customer_age").cast("int").alias("customer_age")
    )
    .filter(F.col("customer_email").isNotNull())
    .dropDuplicates(["customer_email"])
    .withColumn(
        "customer_id",
        F.md5(F.col("customer_email"))
    )
    .select(
        "customer_id",
        "customer_name",
        "customer_email",
        "customer_phone",
        "customer_age"
    )
)

display(df_dim_customer.limit(10))

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6c146a9f-028e-4da2-a205-74a77b7ea546)

In [5]:
#Parse items để tạo thành từng dòng riêng
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DecimalType, ArrayType
)

# Khai báo schema cho cột items
items_schema = ArrayType(
    StructType([
        StructField("product_id", StringType(), True),
        StructField("category", StringType(), True),
        StructField("price", DecimalType(12, 2), True),
        StructField("quantity", IntegerType(), True)
    ])
)

# Parse và explode items
df_sales_items = (
    df_sales
    .withColumn(
        "items_parsed",
        F.from_json(F.col("items"), items_schema)
    )
    .withColumn(
        "item",
        F.explode(F.col("items_parsed"))
    )
    .select(
        "order_id",
        F.col("item.product_id").alias("product_id"),
        F.col("item.category").alias("category"),
        F.col("item.price").alias("unit_price"),
        F.col("item.quantity").alias("quantity")
    )
)

display(df_sales_items.limit(10))

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fc62ea96-b03d-4a85-98a5-780f98ec5e9b)

In [6]:
# Tạo DataFrame dim_product
df_dim_product = (
    df_sales_items
    .select(
        F.trim(F.col("product_id")).alias("product_id"),
        F.trim(F.col("category")).alias("category")
    )
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
)

display(df_dim_product.limit(10))

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 48ab1c76-c3e1-45bd-8aa8-9e158909988a)

In [7]:
# Tạo DataFrame fact_sales
df_fact_sales = (
    df_sales
    .withColumn("date_key", F.date_format(F.to_date(F.col("order_date")), "yyyyMMdd").cast("int"))
    .withColumn("customer_id", F.md5(F.lower(F.trim(F.col("customer_email")))))
    .withColumn("location_id", F.md5(F.lower(F.trim(F.col("location")))))
    .withColumn("year", F.year(F.col("order_date")))
    .withColumn("month", F.month(F.col("order_date")))
)

# Join với bảng exchange_rate
df_fact_sales = (
    df_fact_sales.alias("s")
    .join(
        df_exchange_rate.alias("e"),
        (F.col("s.year") == F.col("e.year")) &
        (F.col("s.month") == F.col("e.month")) &
        (F.col("s.currency") == F.col("e.from_currency")) &
        (F.col("e.to_currency") == F.lit("VND")),
        how="left"
    )
    .withColumn(
        "total_amount_vnd",
        F.round(
            F.col("s.total_amount").cast(DecimalType(12, 2)) * F.col("e.exchange_rate").cast(DecimalType(18, 4)),0
        )
    )
)

# Chọn cột đưa vào DataFrame
df_fact_sales = df_fact_sales.select(
    F.col("s.order_id").alias("order_id"),
    F.col("s.date_key").alias("date_key"),
    F.col("s.customer_id").alias("customer_id"),
    F.col("s.location_id").alias("location_id"),
    F.col("s.payment_method").alias("payment_method"),
    F.col("s.currency").alias("currency"),
    F.col("s.discount_code").alias("discount_code"),
    F.col("s.shipping_cost").cast(DecimalType(12, 2)).alias("shipping_cost"),
    F.col("s.total_amount").cast(DecimalType(12, 2)).alias("total_amount"),
    F.col("e.exchange_rate").cast(DecimalType(18, 4)).alias("exchange_rate"),
    F.col("total_amount_vnd").cast(DecimalType(18, 2)).alias("total_amount_vnd"),
    F.col("s.order_status").alias("order_status"),
    F.col("s.loyalty_points").cast("int").alias("loyalty_points"),
    F.col("s.feedback_score").cast("int").alias("feedback_score")
)

display(df_fact_sales.limit(10))

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ad295f90-51a6-4bb2-b687-1ebdd30b80fe)

In [8]:
# Tạo DataFrame fact_order_items
df_fact_order_items = (
    df_sales_items
    .withColumn(
        "subtotal",
        (
            F.col("unit_price").cast(DecimalType(12, 2)) *
            F.col("quantity").cast("int")
        ).cast(DecimalType(12, 2))
    )
    .select(
        "order_id",
        "product_id",
        "category",
        F.col("unit_price").cast(DecimalType(12, 2)).alias("unit_price"),
        F.col("quantity").cast("int").alias("quantity"),
        "subtotal"
    )
)

display(df_fact_order_items.limit(10))

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ce3b5e05-be7b-44fa-b496-a17d1544a1ab)

In [10]:
# # Lưu vào Gold
tables = {
    "gold_dim_date"       : df_dim_date,
    "gold_dim_customer"   : df_dim_customer,
    "gold_dim_location"   : df_dim_location,
    "gold_dim_product"    : df_dim_product,
    "gold_fact_sales"     : df_fact_sales,
    "gold_fact_order_items": df_fact_order_items,
}

for table_name, df in tables.items():
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 12, Finished, Available, Finished, False)

In [11]:
# Tạo bảng lọc bỏ dữ liệu ảo
df_valid_orders = df_fact_sales.filter(
    (F.col("order_status") != "Failed") &
    (
        F.col("feedback_score").isNull() |
        F.col("feedback_score").isin(1, 2, 3, 4, 5)
    )
)

display(df_valid_orders.limit(30))

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a9d663ea-15a3-469c-8b9b-872fd32adc2d)

In [14]:
# Doanh thu VND hàng tháng cho từng danh mục sản phẩm
df_gold_report_monthly_revenue_vnd = (
    df_valid_orders
    .join(df_fact_order_items, "order_id", "inner")
    .withColumn("year", F.col("date_key").cast("string").substr(1, 4).cast("int"))
    .withColumn("month", F.col("date_key").cast("string").substr(5, 2).cast("int"))
    .join(
        df_exchange_rate.select(
            "year",
            "month",
            F.col("exchange_rate").alias("exchange_rate_gold")
        ),
        on=["year", "month"],
        how="left"
    )
    .withColumn(
        "revenue_vnd",
        F.round(
            F.col("subtotal").cast(DecimalType(12, 2)) * F.col("exchange_rate_gold").cast(DecimalType(18, 4)),0
        )
    )
    .groupBy("year", "month", "category")
    .agg(
        F.round(F.sum("revenue_vnd"), 0).alias("Total_Revenue_VND"),
        F.countDistinct("order_id").alias("order_count"),
        F.round(F.avg("exchange_rate_gold"), 0).alias("avg_exchange_rate")
    )
    .orderBy("year", "month", "category")
)

display(df_gold_report_monthly_revenue_vnd.limit(30))

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e24f3df1-db38-47e4-86fe-9ce0a5d2649b)

In [16]:
# Tỷ lệ phần trăm số đơn hàng có sử dụng mã giảm giá phân bổ theo từng khu vực địa lý
df_gold_discount_by_location = (
    df_valid_orders
    .join(
        df_dim_location,
        on="location_id",
        how="left"
    )
    .withColumn(
        "has_discount",
        F.when(
            F.col("discount_code").isNotNull() &
            (F.trim(F.col("discount_code")) != ""),
            1
        ).otherwise(0)
    )
    .groupBy("location_id", "city")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("has_discount").alias("discount_orders")
    )
    .withColumn(
        "discount_order_pct",
        F.round(
            F.col("discount_orders") / F.col("total_orders") * 100,
            2
        )
    )
    .orderBy(F.col("discount_order_pct").desc())
)

display(df_gold_discount_by_location.limit(30))

StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ffbb7734-616f-4b9c-9d7c-069c366d6b18)

In [17]:
# Lưu vào bảng Gold tính toán chỉ số kinh doanh
business_tables = {
    "gold_report_monthly_revenue_vnd": df_gold_report_monthly_revenue_vnd,
    "gold_discount_by_location": df_gold_discount_by_location
}

for table_name, df in business_tables.items():
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)


StatementMeta(, 16c242a8-bd42-4070-b2a0-2a3497a3c6bd, 19, Finished, Available, Finished, False)